# Load Dataset

In [1]:
################################################################################
# Load dataset and split it into training and test set (multi-class)
################################################################################

import pandas as pd
import os
from tabulate import tabulate

dataset_name = "cic-iot"
sample_size = 100000

# Load dataset
df = pd.read_csv(os.getcwd() + f'/data/sample-{sample_size}-2.csv')
# Keep original multi-class labels — NO binary merging

train_dfs, test_dfs = {}, {}
for label, group in df.groupby('label'):
    train_part = group.sample(frac=0.8, random_state=42)
    test_part  = group.drop(train_part.index)
    train_dfs[label] = train_part
    test_dfs[label]  = test_part

df_train = pd.concat(train_dfs.values())
df_test  = pd.concat(test_dfs.values())

X_train = df_train.drop(columns=['label'])
y_train = df_train['label']
X_test  = df_test.drop(columns=['label'])
y_test  = df_test['label']

# Table: class | total | train | test
table_data = [[lbl, len(group), len(train_dfs[lbl]), len(test_dfs[lbl])]
              for lbl, group in df.groupby('label')]
print(tabulate(table_data, headers=["Class", "Total", "Train", "Test"], tablefmt="grid"))


+-------------------------+---------+---------+--------+
| Class                   |   Total |   Train |   Test |
+=========================+=========+=========+========+
| Backdoor_Malware        |       1 |       1 |      0 |
+-------------------------+---------+---------+--------+
| BenignTraffic           |     304 |     243 |     61 |
+-------------------------+---------+---------+--------+
| DDoS-ACK_Fragmentation  |       3 |       2 |      1 |
+-------------------------+---------+---------+--------+
| DDoS-ICMP_Flood         |      59 |      47 |     12 |
+-------------------------+---------+---------+--------+
| DDoS-ICMP_Fragmentation |       2 |       2 |      0 |
+-------------------------+---------+---------+--------+
| DDoS-PSHACK_Flood       |      24 |      19 |      5 |
+-------------------------+---------+---------+--------+
| DDoS-RSTFINFlood        |      25 |      20 |      5 |
+-------------------------+---------+---------+--------+
| DDoS-SYN_Flood          |    

# ML Baseline (Multi-Class)

In [2]:
################################################################################
# Decision Tree and Random Forest (multi-class)
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import os
import time

os.makedirs("results/ml", exist_ok=True)

for Model, fname in [(DecisionTreeClassifier, "result-dt-multiclass-100000.txt"),
                     (RandomForestClassifier,  "result-rf-multiclass-100000.txt")]:
    model = Model(random_state=42)
    model.fit(X_train, y_train)
    start = time.time()
    y_pred = model.predict(X_test)
    elapsed = time.time() - start
    report = classification_report(y_true=y_test, y_pred=y_pred, digits=4)
    matrix = confusion_matrix(y_test, y_pred)
    print(f"=== {Model.__name__} ({elapsed:.4f}s) ===")
    print(report)
    with open(f"results/ml/{fname}", "w") as f:
        f.write(f"Classification Report\n{report}\n\nConfusion Matrix\n{matrix}\n")


=== DecisionTreeClassifier (0.0005s) ===
                         precision    recall  f1-score   support

          BenignTraffic     0.9833    0.9672    0.9752        61
 DDoS-ACK_Fragmentation     1.0000    1.0000    1.0000         1
        DDoS-ICMP_Flood     1.0000    1.0000    1.0000        12
      DDoS-PSHACK_Flood     1.0000    1.0000    1.0000         5
       DDoS-RSTFINFlood     1.0000    1.0000    1.0000         5
         DDoS-SYN_Flood     1.0000    1.0000    1.0000         5
DDoS-SynonymousIP_Flood     1.0000    1.0000    1.0000         6
         DDoS-TCP_Flood     1.0000    1.0000    1.0000         5
         DDoS-UDP_Flood     1.0000    1.0000    1.0000         6
 DDoS-UDP_Fragmentation     0.0000    0.0000    0.0000         1
          DoS-SYN_Flood     1.0000    1.0000    1.0000         3
          DoS-TCP_Flood     1.0000    1.0000    1.0000         4
          DoS-UDP_Flood     1.0000    1.0000    1.0000         6
       MITM-ArpSpoofing     0.0000    0.0000    

/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/S4160163/Documents/Projects/RAG Paper/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_pr

# Vector Store — Per-Class Representative Samples

In [1]:
################################################################################
# Build class_entries dict with representative samples per class
################################################################################

import json
import numpy as np
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm

embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-m3')
feature_cols = X_train.columns.tolist()
n_results = 10
max_embed_per_class = 100   # cap to avoid slow startup

class_entries = {}
for label in tqdm(df['label'].unique(), desc="Building class entries"):
    class_df = train_dfs[label].drop(columns=['label'])
    sample_df = class_df.sample(n=min(max_embed_per_class, len(class_df)), random_state=42)
    docs = [str(row.tolist()) for _, row in sample_df.iterrows()]
    # Embed docs
    vecs = np.array(embeddings.embed_documents(docs))
    mean_vec = vecs.mean(axis=0)
    # Cosine similarities to mean
    norms = np.linalg.norm(vecs, axis=1) * np.linalg.norm(mean_vec)
    sims = (vecs @ mean_vec) / np.where(norms == 0, 1e-9, norms)
    top_idx = np.argsort(sims)[::-1][:n_results]
    top_rows = sample_df.iloc[top_idx]
    # Transpose to feature-keyed dict
    entries = {}
    for col in feature_cols:
        entries[col] = top_rows[col].tolist()
    class_entries[label] = entries

print("class_entries keys:", list(class_entries.keys()))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

# Tool Definition

In [4]:
################################################################################
# Tool: evaluate_rule (multi-class aware)
################################################################################

from sklearn.metrics import classification_report
from tqdm import tqdm
import operator
from typing import Annotated
from langchain_core.tools import tool

show_progress = False
operations = {'<': operator.lt, '>': operator.gt, '==': operator.eq,
              '<=': operator.le, '>=': operator.ge, '!=': operator.ne}

@tool
def evaluate_rule(
    feature_name: Annotated[str, "Feature name"],
    value:        Annotated[str, "Value"],
    op:           Annotated[str, "Operator"],
    target_class: Annotated[str, "Target class name"]
) -> float:
    """Evaluate rule on training set. Returns macro F1 for target_class vs all others."""
    try:
        value = float(value)
    except ValueError:
        pass
    if op not in operations:
        raise ValueError(f"Unsupported operator: {op}")
    y_true, y_pred = [], []
    for lbl, df_cls in train_dfs.items():
        df_feat = df_cls.drop(columns=['label'])
        for i in tqdm(range(len(df_feat)), disable=not show_progress,
                      desc=f"Eval {lbl[:12]}..."):
            y_true.append("target" if lbl == target_class else "other")
            y_pred.append("target" if operations[op](df_feat.iloc[i][feature_name], value) else "other")
    report = classification_report(y_true, y_pred, digits=4, output_dict=True, zero_division=0)
    return report['macro avg']['f1-score']


# Prompt Template

In [5]:
################################################################################
# Prompt Template
################################################################################

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = ("system",
"""You are a good data analyst. You are provided with network data entries from multiple traffic \
classes along with feature names. Carefully analyze differences between classes. \
Your task is to generate {k} simple and deterministic rules for the top {k} important features \
to filter entries of the target class '{target_class}'. \
Supported operators are '==', '!=', '>', '<', '>=', '<='. \
Generate exactly {k} rules and make a tool call for each rule."""
)

human_message = ("user",
"""Analyze the following network data and generate rules for the top {k} important features \
to identify '{target_class}' entries.

Target Class Entries:
```{target_entries}```

Other Class Entries (sample):
```{other_entries}```"""
)

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder("msgs")
])


# LLM Setup

In [9]:
################################################################################
# LLM Setup and Tool Binding
################################################################################

import os
import dotenv
# from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

dotenv.load_dotenv(os.getcwd() + '/../.env')

# model_name = "gpt-4o"
model_name = "claude-haiku-4-5"
# llm = ChatOpenAI(model=model_name, temperature=0.1)
llm = ChatAnthropic(model=model_name, temperature=0.1)

llm_with_tool = llm.bind_tools([evaluate_rule])


# Feedback Loop (Per Class)

In [8]:
################################################################################
# Per-class feedback loop
################################################################################

import json
from langchain_core.messages import HumanMessage

chain = prompt | llm_with_tool

n_repetitions = 5
show_progress = False
class_rules = {}   # class_name -> final list of tool_call dicts

non_benign_classes = [lbl for lbl in df['label'].unique() if lbl != 'BenignTraffic']

for target_class in non_benign_classes:
    print(f"\n=== Target class: {target_class} ===")
    n, k = 0, 5
    mean_f1s = 0
    token_usage = {}
    target_entries = json.dumps(class_entries[target_class])
    other_entries  = json.dumps(class_entries['BenignTraffic'])
    msgs = []
    train_f1_scores = []

    while n < n_repetitions:
        ai_msg = chain.invoke({
            "k": k, "target_class": target_class,
            "target_entries": target_entries, "other_entries": other_entries,
            "msgs": msgs
        })
        tool_msgs = []
        for tool_call in ai_msg.tool_calls:
            tool_msg = evaluate_rule.invoke(tool_call)
            tool_msgs.append(tool_msg)

        if tool_msgs:
            mean_f1s = sum(float(m.content) for m in tool_msgs) / len(tool_msgs)
        human_msg = HumanMessage(
            f"The current mean f1-score for the generated rules is {mean_f1s}. "
            "If this mean f1-score is greater than the previous rounds, keep the better performing "
            "rules and revise or replace only the underperforming ones (those with a score less than mean). "
            "Otherwise, revise or replace any rules that have a score less than mean. "
            f"Based on the feedback, generate exactly {k} rules to identify '{target_class}' entries "
            "and make a tool call for each rule, ensuring that a tool call is made for every entry every time."
        )
        n += 1
        # msgs.extend([ai_msg, *tool_msgs, human_msg])
        msgs.extend([ai_msg, *tool_msgs, human_msg])

        train_f1_scores.append(mean_f1s)

        # token_usage = {key: ai_msg.response_metadata["token_usage"][key]
                    #    for key in ["completion_tokens", "prompt_tokens", "total_tokens"]}
        # print(f"  Round: {n}  Mean F1: {mean_f1s:.4f}  Tokens: {token_usage}")

        # Extract token usage - Claude format
        usage_info = ai_msg.response_metadata.get("usage", {})
        token_usage = {
            "completion_tokens": usage_info.get("output_tokens", 0),
            "prompt_tokens": usage_info.get("input_tokens", 0),
            "total_tokens": usage_info.get("input_tokens", 0) + usage_info.get("output_tokens", 0)
        }
        print(f"  Round: {n}  Mean F1: {mean_f1s:.4f}  Tokens: {token_usage}")


    # Save final tool_calls from last AI message
    final_tool_calls = [tc for msg in msgs if msg.type == "ai"
                        for tc in (msg.additional_kwargs.get("tool_calls") or [])]
    # Keep only the most recent k calls
    class_rules[target_class] = final_tool_calls[-k:] if len(final_tool_calls) >= k else final_tool_calls
    print(f"  Train F1 history: {train_f1_scores}")

print("\nDone. Classes with rules:", list(class_rules.keys()))



=== Target class: DDoS-UDP_Flood ===
  Round: 1  Mean F1: 0.6317  Tokens: {'completion_tokens': 923, 'prompt_tokens': 5424, 'total_tokens': 6347}
  Round: 2  Mean F1: 0.6391  Tokens: {'completion_tokens': 828, 'prompt_tokens': 6655, 'total_tokens': 7483}
  Round: 3  Mean F1: 0.7203  Tokens: {'completion_tokens': 854, 'prompt_tokens': 7791, 'total_tokens': 8645}
  Round: 4  Mean F1: 0.7899  Tokens: {'completion_tokens': 879, 'prompt_tokens': 8951, 'total_tokens': 9830}
  Round: 5  Mean F1: 0.8012  Tokens: {'completion_tokens': 888, 'prompt_tokens': 10135, 'total_tokens': 11023}
  Train F1 history: [0.6317162563308582, 0.6390517087501049, 0.7202543454597026, 0.789853234642081, 0.8011601111901951]

=== Target class: DDoS-RSTFINFlood ===
  Round: 1  Mean F1: 0.7643  Tokens: {'completion_tokens': 757, 'prompt_tokens': 5457, 'total_tokens': 6214}
  Round: 2  Mean F1: 0.7695  Tokens: {'completion_tokens': 760, 'prompt_tokens': 6519, 'total_tokens': 7279}
  Round: 3  Mean F1: 0.8715  Tokens: 

KeyboardInterrupt: 

# Evaluate All Rules on Test Set (Multi-Class)

In [ ]:
################################################################################
# Evaluate generated rules on test set (multi-class)
################################################################################

import json
import operator
import os
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

operations = {'<': operator.lt, '>': operator.gt, '==': operator.eq,
              '<=': operator.le, '>=': operator.ge, '!=': operator.ne}

os.makedirs("results/llm", exist_ok=True)

def predict_multiclass(row, class_rules):
    votes = {}   # class -> count of rules that fired
    for cls, tool_calls in class_rules.items():
        count = 0
        for tc in tool_calls:
            args = json.loads(tc["function"]["arguments"])
            op, feat, val = args["op"], args["feature_name"], args["value"]
            try:
                val = float(val)
            except ValueError:
                pass
            if op in operations and feat in row.index:
                if operations[op](row[feat], val):
                    count += 1
        votes[cls] = count
    best_cls = max(votes, key=lambda c: votes[c]) if votes else 'BenignTraffic'
    return best_cls if votes.get(best_cls, 0) > 0 else 'BenignTraffic'

y_pred_llm, y_true_llm = [], []
for i in tqdm(range(len(X_test)), desc="Evaluating multiclass rules"):
    row = X_test.iloc[i]
    y_true_llm.append(y_test.iloc[i])
    y_pred_llm.append(predict_multiclass(row, class_rules))

report = classification_report(y_true_llm, y_pred_llm, digits=4)
matrix = confusion_matrix(y_true_llm, y_pred_llm)
print(report)
with open("results/llm/result-llm-multiclass-100000.txt", "w") as f:
    f.write(f"Classification Report\n{report}\n\nConfusion Matrix\n{matrix}\n")


# Efficiency Comparison

In [ ]:
################################################################################
# Efficiency Comparison
################################################################################

import time
from tabulate import tabulate

# Retrain DT and RF for timing
model_dt = DecisionTreeClassifier(random_state=42)
model_rf = RandomForestClassifier(random_state=42)
model_dt.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

elapsed_dt, elapsed_rf, elapsed_llm = [], [], []
n_samples = min(1000, len(X_test))   # cap to avoid slow LLM rules

for i in range(n_samples):
    row = X_test.iloc[[i]]

    t0 = time.time()
    model_dt.predict(row)
    elapsed_dt.append(time.time() - t0)
    
    t0 = time.time()
    model_rf.predict(row)
    elapsed_rf.append(time.time() - t0)
    
    t0 = time.time()
    predict_multiclass(X_test.iloc[i], class_rules)
    elapsed_llm.append(time.time() - t0)

rows = [
    ["Decision Tree", f"{sum(elapsed_dt)/n_samples*1e3:.4f} ms"],
    ["Random Forest", f"{sum(elapsed_rf)/n_samples*1e3:.4f} ms"],
    ["LLM Rules",     f"{sum(elapsed_llm)/n_samples*1e3:.4f} ms"],
]
print(tabulate(rows, headers=["Model", "Avg Inference Time"], tablefmt="grid"))
